In [40]:
import os
import re
import json
import yaml
# from langchain.chat_models import AzureChatOpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from dotenv import load_dotenv

In [37]:
load_dotenv()

# Access the environment variables
os.environ["AZURE_OPENAI_API_KEY"] = os.getenv("AZURE_OPENAI_API_KEY")
os.environ["AZURE_OPENAI_ENDPOINT"] = os.getenv("AZURE_OPENAI_ENDPOINT")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

In [ ]:
load_dotenv()

llm = ChatOpenAI(
    model="gpt-5-mini",
    temperature=1,
    max_tokens=8192,
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_API_BASE")
)

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_API_BASE")
)

In [39]:
print(llm.invoke("Hello, how are you?").content)

I'm an AI so I don't have feelings, but I'm here and ready to help. What can I do for you today?


In [42]:
# llm = AzureChatOpenAI(
#       temperature=0.5,
#       model="gpt-4o",
#       openai_api_version="2024-02-01",
#       azure_deployment="gpt4o",
#       max_tokens=1024)

# embedding_model = AzureOpenAIEmbeddings(
#     azure_deployment="text-embedding-3-small",  # Your Azure deployment name for embeddings
#     openai_api_version="2024-02-01",
#     azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
#     api_key=os.environ["AZURE_OPENAI_API_KEY"]
# )

In [ ]:
# from langchain.vectorstores import FAISS
# import faiss
# import pickle
# from langchain_community.docstore.in_memory import InMemoryDocstore

# index = faiss.read_index("./data/vector_data/faiss_index")

# with open("./data/vector_data/faiss_documents.pkl", "rb") as f:
#     docstore_dict = pickle.load(f)

# vector_store_new = FAISS(
#     embedding_function=embedding_model,
#     index=index,
#     docstore=InMemoryDocstore(docstore_dict),
#     index_to_docstore_id=dict(zip(range(len(docstore_dict)), docstore_dict.keys()))
# )

In [43]:
vector_store = Chroma(
    persist_directory="./app/data/vector_data/chroma_db",
    collection_name="policy_docs",
    embedding_function=embedding_model,   # IMPORTANT: must be same embedding model
)

In [44]:
retriever = vector_store.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 5}
    )
    
res = retriever.invoke("Best policy for older people")

In [46]:
prompts = yaml.safe_load(open("./app/src/utils/prompt_templates.yaml"))

In [47]:
def clean_json_response(response: str) -> str:
    """Extracts valid JSON from an LLM response, removing markdown formatting and extra text."""
    match = re.search(r"```json\n(.*?)\n```", response, re.DOTALL)
    return match.group(1) if match else response.strip()

def restructure_query(query: str) -> dict:
    prompt = prompts['adaptive_rag_prompt'].format(original_query=query)

    sub_queries_response = llm.invoke(prompt).content
    try:
        cleaned_response = clean_json_response(sub_queries_response)
        return json.loads(cleaned_response)
    except json.JSONDecodeError:
        print("Error: Invalid JSON from LLM!")
        return {}

In [48]:
# # query = "Waiting period of pre existing disease of all the plans"
# query = "Which Plan is best for older people"
# res = restructure_query(query)

In [58]:
def get_relevant_doc(policy, plan, query):
    filter_conditions = []
    if policy:
        filter_conditions.append({"company": {"$eq": policy}})
    if plan:
        filter_conditions.append({"plan": {"$eq": plan}})
    
    # Apply filter correctly
    if len(filter_conditions) > 1:
        combined_filter = {"$and": filter_conditions}
    elif len(filter_conditions) == 1:
        combined_filter = filter_conditions[0]  # Use single condition directly
    else:
        combined_filter = None  # No filter
    
    retriever = vector_store.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 5, "filter": combined_filter}
    )
    
    return retriever.invoke(query)

def get_relevant_docs(res):
    docs = []
    if res.get('query'):
        docs.extend(get_relevant_doc(
            policy=res.get('policy'), 
            plan=res.get('plan'), 
            query=res.get('query')
        ))
    else:
        for v in res.values():
            docs.extend(get_relevant_doc(
                policy=v.get('policy'), 
                plan=v.get('plan'), 
                query=v.get('query')
            ))

    doc_entries = [
        f'"company": {doc.metadata["company"]} \n "plan": {doc.metadata["plan"]} \n "content": {doc.page_content}' for doc in docs
    ]

    return doc_entries

In [50]:
def refine_document_with_llm(query: str, retrieved_doc: str) -> str:
    prompt = prompts['corrective_rag_prompt'].format(
        original_query=query,
        retrieved_doc=retrieved_doc
    )

    res = llm.invoke(prompt).content.strip()

    return res

def refine_documents_with_llm(query, doc_entries):
    new_doc_entries = []

    for doc_entry in doc_entries:
        res = refine_document_with_llm(query, doc_entry)
        if int(res) >= 2:
            new_doc_entries.append(doc_entry)
        # else:
        #     print(doc_entry)

    new_doc_entries_str = "\n\n".join(new_doc_entries)
    return new_doc_entries_str

In [51]:
def final_content_refinement_tool(query, retrieved_docs):
    prompt= prompts['final_answer_generation_prompt'].format(
        original_query=query,
        retrieved_docs=retrieved_docs
    )
    
    res = llm.invoke(prompt).content.strip()
    return res


In [52]:
def GradeHallucinations(documents: str, generation: str) -> str:
    """
    Grades a student's answer based on given FACTS.
    
    - Score 1: The student's answer is grounded in the FACTS.
    - Score 0: The student's answer contains hallucinated information.
    
    Returns "0" or "1" as a string.
    """
    
    prompt = prompts['hallucination_checker_prompt'].format(
        docs=documents,
        answer=generation
    )

    return llm.invoke(prompt).content.strip()

In [59]:
def generate_response(query):
    """
    Generates a response based on the query using a multi-step RAG pipeline.
    The response is saved to a Markdown file if it passes the hallucination check.
    """
    # Step 1: Restructure the query for better retrieval
    refined_query = restructure_query(query)
    print("Step 1: Refined Query completed.")
    
    # Step 2: Retrieve relevant documents based on the refined query
    relevant_docs = get_relevant_docs(refined_query)
    print("Step 2: Document Retrieval completed.")

    # Step 3: Refine the retrieved documents using an LLM
    refined_docs = refine_documents_with_llm(query, relevant_docs)
    print("Step 3: Document Refinement completed.")

    # Step 4: Generate the final response based on the refined documents
    response = final_content_refinement_tool(query, refined_docs)
    print("Step 4: Final Response Generation completed.")

    # Step 5: Check if the response is based on the retrieved documents
    if GradeHallucinations(refined_docs, response):
        safe_query = re.sub(r'[<>:"/\\|?*]', '_', query)
        output_dir = "./app/data/generated_output"
        os.makedirs(output_dir, exist_ok=True)  # Ensure directory exists
        output_path = os.path.join(output_dir, f"{safe_query}.md")
        
        with open(output_path, "w", encoding="utf-8") as file:
            file.write(response)
        print(f"Success: Response saved to {output_path}")
    else:
        print("Error: The generated response is not factually grounded.")


In [60]:
query = "Does HDFC ERGO optima secure covers AYUSH treatment?"
generate_response(query)

Step 1: Refined Query completed.
Step 2: Document Retrieval completed.
Step 3: Document Refinement completed.
Step 4: Final Response Generation completed.
Success: Response saved to ./app/data/generated_output/Does HDFC ERGO optima secure covers AYUSH treatment_.md


In [36]:
from langchain_community.tools.tavily_search import TavilySearchResults
web_search_tool = TavilySearchResults(k=3)

In [ ]:
# Web search
query = "All details about CARE Supre OPD treatment"
docs = web_search_tool.invoke({"query": query})

In [38]:
docs

[{'title': 'Care Supreme Must-look Features & Claims Settlement Ratio',
  'url': 'https://1finance.co.in/product-scoring/health-insurance/care-supreme?gender=Male&age=46-50&family=self-&sum=10%20Lacs',
  'content': '... Finance: Explore the essential must-look features of Care Supreme and delve into their claims settlement ratio ... Claim Settlement Ratio (CSR) - Number92.81%.',
  'score': 0.91563576},
 {'title': 'Care Insurance Care Supreme Plan Features, Benefits, Review ...',
  'url': 'https://www.beshak.org/insurance/health-insurance/best-health-insurance-plans/care-insurance-care-supreme/',
  'content': 'Care Insurance Health Insurance Company has a claim settlement ratio of 90.50%. Network hospitals: Care Insurance Health Insurance Company has',
  'score': 0.7613660444444444},
 {'title': 'Care Health Insurance UPDATED Review 2025 - YouTube',
  'url': 'https://www.youtube.com/watch?v=gnJyocoSiMA',
  'content': "With a claim settlement ratio of 90% and updated operational metrics, 